# 3.33 — Ordinal & Multi-Output Regression

Ordinal and multi-output regression are two ways of respecting structure in targets instead of pretending every prediction is one unrelated scalar. Ordinal regression remembers that labels such as 1, 2, 3, 4, 5 have an order, while multi-output regression remembers that several target components for the same example can share signal, share cost, and be judged by an averaged loss.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build ordinal and multi-output regression one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is exposed with small arrays so the loss, cost, validation gap, and final decision are inspectable. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, linear algebra, thresholds, and small numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for synthetic target examples.

### 1. Ordered labels are not just class names

Ordinal regression starts with labels that have a real order: level 1 is below level 2, level 2 is below level 3, and so on. The first modeling decision is to preserve that geometry. Predicting level 4 when the truth is 5 should be a smaller mistake than predicting level 1 when the truth is 5, even though both are "wrong" as class labels.

In [ ]:
y_ord_w = np.array([1, 2, 3, 4, 5])  # five ordered satisfaction levels.
yhat_near_w = np.array([1, 2, 3, 4, 4])  # one adjacent miss at the top.
yhat_far_w = np.array([1, 1, 1, 1, 1])  # several far misses.
print("truth:", y_ord_w)
print("near predictions:", yhat_near_w)
print("far predictions:", yhat_far_w)

▶ What you'll see: the two prediction vectors contain different kinds of mistakes even when a plain accuracy view only sees mismatches.

In [ ]:
abs_near_w = np.abs(y_ord_w - yhat_near_w)  # ordinal distance for each example.
abs_far_w = np.abs(y_ord_w - yhat_far_w)  # larger distances for farther mistakes.
print("near absolute errors:", abs_near_w, "mean =", round(abs_near_w.mean(), 3))
print("far absolute errors:", abs_far_w, "mean =", round(abs_far_w.mean(), 3))
assert round(abs_near_w.mean(), 3) == 0.200
assert round(abs_far_w.mean(), 3) == 2.000

▶ What you'll see: the adjacent-miss model has mean ordinal error 0.2, while the collapsed-low model has mean ordinal error 2.0.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(y_ord_w, marker="o", label="truth")
plt.plot(yhat_near_w, marker="s", label="near")
plt.plot(yhat_far_w, marker="^", label="far")
plt.title("1: ordinal labels live on a line")
plt.xlabel("example")
plt.ylabel("ordered level")
plt.legend()
plt.show()

▶ What you'll see: the far predictions sit much farther down the ordered scale, not merely in a different category.

*Why it's done this way:* ordinal losses encode distance on the target scale. A one-step miss and a four-step miss should not receive the same penalty because the order itself carries information about how bad the error is.

### 2. Thresholds turn one score into ordered classes

A simple ordinal model can learn one continuous score and then cut that score with ordered thresholds. With thresholds at 1.5, 2.5, 3.5, and 4.5, a score below 1.5 becomes level 1, a score between 1.5 and 2.5 becomes level 2, and so on. The thresholds are the bridge from regression-like scores to ordered labels.

In [ ]:
scores_w = np.array([1.2, 1.9, 2.8, 3.7, 4.6])  # one latent severity/satisfaction score per example.
thresholds_w = np.array([1.5, 2.5, 3.5, 4.5])  # ordered cut points.
levels_w = 1 + np.sum(scores_w[:, None] > thresholds_w[None, :], axis=1)  # count thresholds passed.
print("scores:", scores_w)
print("predicted ordinal levels:", levels_w)
assert np.array_equal(levels_w, np.array([1, 2, 3, 4, 5]))

▶ What you'll see: counting how many thresholds a score passes recovers the ordered level.

In [ ]:
plt.figure(figsize=(5, 3))
plt.scatter(scores_w, levels_w, s=80, color="teal")
for t_w in thresholds_w:
    plt.axvline(t_w, color="gray", linestyle="--", linewidth=1)
plt.title("2: thresholds carve a score line into levels")
plt.xlabel("latent score")
plt.ylabel("predicted level")
plt.yticks([1, 2, 3, 4, 5])
plt.show()

▶ What you'll see: vertical threshold lines split one numeric axis into five ordered regions.

*Why it's done this way:* one latent score preserves monotonic order, while thresholds define where the decision changes. This is more structured than training five unrelated classes because moving the score upward can only move the prediction upward through adjacent levels.

### 3. Multi-output targets share one example and one loss

Multi-output regression predicts a vector \(\hat y=(\hat y_1,\ldots,\hat y_q)\) for each example. The lesson's core loss averages component losses, \(L=\frac1q\sum_{j=1}^q \ell(y_j,\hat y_j)\), so no single target component silently becomes the whole objective.

In [ ]:
Y_w = np.array([[3.0, 8.0, 1.0], [4.0, 7.0, 2.0], [5.0, 6.0, 3.0]])  # three examples, q=3 outputs.
Yhat_w = np.array([[2.8, 8.2, 1.2], [4.3, 6.6, 1.7], [4.4, 6.4, 3.5]])  # vector predictions.
component_sq_w = (Y_w - Yhat_w) ** 2  # squared loss per output component.
print("component squared losses:\n", np.round(component_sq_w, 3))
print("per-example vector losses:", np.round(component_sq_w.mean(axis=1), 3))

▶ What you'll see: each row has three component errors, then one averaged vector loss.

In [ ]:
losses_w = component_sq_w.mean(axis=1)
R_w = float(losses_w.mean())
print("empirical multi-output risk:", round(R_w, 3))
assert round(R_w, 3) == 0.137

▶ What you'll see: the risk is an average over outputs and examples, matching the lesson's averaging move.

In [ ]:
plt.figure(figsize=(5, 3))
plt.imshow(component_sq_w, cmap="magma", aspect="auto")
plt.colorbar(label="squared error")
plt.title("3: multi-output loss has component structure")
plt.xlabel("output component")
plt.ylabel("example")
plt.show()

▶ What you'll see: the heatmap shows which output component drives each example's vector loss.

*Why it's done this way:* averaging across \(q\) outputs gives a single ERM objective while preserving the fact that each prediction is a vector. The division by \(q\) keeps the loss scale comparable when the number of outputs changes.

### 4. Shared weights can exploit related outputs

If outputs are related, a shared representation can be more stable than fitting each target in isolation. In the smallest linear case, all outputs use the same feature matrix \(X\), and a weight matrix \(W\) maps features to multiple targets with \(\hat Y=XW\). Each column is still one output, but fitting them together makes the shared input geometry explicit.

In [ ]:
X_w = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0], [1.0, 3.0]])  # intercept + one feature.
Y_multi_w = np.array([[2.0, 5.0], [3.0, 4.1], [4.0, 3.0], [5.0, 2.2]])  # two related outputs.
W_w = np.linalg.pinv(X_w) @ Y_multi_w  # least-squares weight matrix, solved with NumPy only.
Yfit_w = X_w @ W_w
print("W shape:", W_w.shape)
print("weights:\n", np.round(W_w, 3))

▶ What you'll see: one 2×2 weight matrix fits both outputs from the same design matrix.

In [ ]:
mse_outputs_w = np.mean((Y_multi_w - Yfit_w) ** 2, axis=0)
print("MSE per output:", np.round(mse_outputs_w, 3))
print("average MSE:", round(float(mse_outputs_w.mean()), 3))
assert np.allclose(np.round(mse_outputs_w, 3), np.array([0.000, 0.004]))

▶ What you'll see: the first output is exactly linear, while the second has a tiny residual.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(X_w[:, 1], Y_multi_w[:, 0], "o", label="target 1")
plt.plot(X_w[:, 1], Yfit_w[:, 0], "-", label="fit 1")
plt.plot(X_w[:, 1], Y_multi_w[:, 1], "s", label="target 2")
plt.plot(X_w[:, 1], Yfit_w[:, 1], "--", label="fit 2")
plt.title("4: one feature matrix predicts two outputs")
plt.xlabel("feature")
plt.ylabel("target value")
plt.legend()
plt.show()

▶ What you'll see: one output rises with the feature while the other falls, both learned through the same matrix equation.

*Why it's done this way:* \(XW\) keeps the shared explanatory variables in one place and lets the columns of \(W\) specialize by output. That is the simplest mathematical version of "related tasks share input evidence but need separate predictions."

### 5. Selection uses raw fit plus cost, not raw fit alone

The lesson's verified toy arithmetic uses three per-example losses, then adds a method cost. That cost may represent complexity, regularization, operational burden, or fragility. The score for selection is not just the pretty training average; it is the full decision score.

In [ ]:
losses533_w = np.array([0.246, 0.083, 0.522])  # verified toy losses from the lesson block.
raw533_w = float(losses533_w.mean())
cost533_w = 0.080
score533_w = raw533_w + cost533_w
print("raw empirical risk:", round(raw533_w, 3))
print("decision score:", round(score533_w, 3))
assert round(raw533_w, 3) == 0.284
assert round(score533_w, 3) == 0.364

▶ What you'll see: the raw average is 0.284, but the score used for selection is 0.364 after adding cost.

In [ ]:
alt533_w = 0.412
gap533_w = alt533_w - score533_w
relative_gap533_w = gap533_w / alt533_w
stable533_w = 0.80 * score533_w
print("gap to flexible alternative:", round(gap533_w, 3))
print("relative gap:", round(relative_gap533_w, 3))
print("stabilized score:", round(stable533_w, 3))
assert round(gap533_w, 3) == 0.048
assert round(relative_gap533_w, 3) == 0.117
assert round(stable533_w, 3) == 0.291

▶ What you'll see: the stabilized score is lower than both the baseline score and the flexible alternative.

In [ ]:
choices533_w = np.array([score533_w, alt533_w, stable533_w])
labels533_w = ["baseline+cost", "flexible", "stabilized"]
plt.figure(figsize=(5, 3))
plt.bar(labels533_w, choices533_w, color=["gray", "orange", "seagreen"])
plt.ylabel("decision score (lower is better)")
plt.title("5: choose by full score")
plt.xticks(rotation=15)
plt.show()

▶ What you'll see: the green stabilized bar is the lowest full decision score.

*Why it's done this way:* ERM supplies the raw average, but model selection must include the cost implied by the method. Otherwise the learner can prefer brittle flexibility just because it made the training fragment look smaller.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, vectorized losses, thresholds, and least squares.
import matplotlib.pyplot as plt  # load Matplotlib for the small diagnostic plots.
np.random.seed(0)  # make all examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Encode ordered labels

**Goal.** Store ordinal targets as numbers with a meaningful order, because level distances matter. We build it in 2 steps.

In [ ]:
y_b1 = np.array([1, 2, 3, 4, 5])  # store ordered levels from low to high.
print("ordinal labels:", y_b1)  # inspect the ordered target vector.

In [ ]:
dist_b1 = y_b1[-1] - y_b1[0]  # compute the distance from lowest to highest level.
print("distance from level 1 to 5:", dist_b1)  # inspect the size of the ordinal scale.
assert dist_b1 == 4
plt.figure(figsize=(4, 3))
plt.scatter(np.arange(len(y_b1)), y_b1, color="teal")
plt.title("Basic 1: ordered target levels")
plt.xlabel("example")
plt.ylabel("level")
plt.show()

▶ What you'll see: the labels form a rising ordered scale, not interchangeable names.

👀 Takeaway: ordinal labels carry distance information that a nominal class code would discard.

### Basic 2 — Penalize nearby and far misses differently

**Goal.** Compare two wrong ordinal predictions, because adjacent mistakes should be less severe than distant mistakes. We build it in 2 steps.

In [ ]:
truth_b2 = np.array([5])  # one high true level.
near_b2 = np.array([4])  # adjacent prediction.
far_b2 = np.array([1])  # distant prediction.
print("truth, near, far:", truth_b2[0], near_b2[0], far_b2[0])

In [ ]:
near_err_b2 = np.abs(truth_b2 - near_b2)  # ordinal distance for the near miss.
far_err_b2 = np.abs(truth_b2 - far_b2)  # ordinal distance for the far miss.
print("near error:", int(near_err_b2[0]), "far error:", int(far_err_b2[0]))
assert int(far_err_b2[0] / near_err_b2[0]) == 4
plt.figure(figsize=(4, 3))
plt.bar(["near", "far"], [near_err_b2[0], far_err_b2[0]], color=["seagreen", "crimson"])
plt.title("Basic 2: ordinal distance")
plt.ylabel("absolute level error")
plt.show()

▶ What you'll see: the far miss has four times the ordinal error of the near miss.

👀 Takeaway: ordinal regression uses target order to grade wrong answers by severity.

### Basic 3 — Convert scores to levels with thresholds

**Goal.** Map continuous scores into ordered levels, because ordinal models often learn a score before applying cut points. We build it in 2 steps.

In [ ]:
scores_b3 = np.array([0.7, 1.8, 2.9, 4.2])  # latent scores from low to high.
thresholds_b3 = np.array([1.5, 2.5, 3.5])  # cut points between four ordered levels.
print("scores:", scores_b3)

In [ ]:
levels_b3 = 1 + np.sum(scores_b3[:, None] > thresholds_b3[None, :], axis=1)  # count thresholds passed.
print("levels:", levels_b3)
assert np.array_equal(levels_b3, np.array([1, 2, 3, 4]))
plt.figure(figsize=(4, 3))
plt.scatter(scores_b3, levels_b3, color="purple", s=80)
for t_b3 in thresholds_b3:
    plt.axvline(t_b3, color="gray", linestyle="--")
plt.title("Basic 3: thresholded scores")
plt.xlabel("score")
plt.ylabel("level")
plt.show()

▶ What you'll see: each threshold crossed raises the predicted level by one.

👀 Takeaway: thresholds preserve monotonic movement along the ordered scale.

### Basic 4 — Compute per-output squared losses

**Goal.** Measure error for each component of a vector target, because multi-output regression predicts several numbers at once. We build it in 2 steps.

In [ ]:
y_b4 = np.array([3.0, 10.0])  # true two-output target.
yhat_b4 = np.array([2.5, 11.0])  # predicted two-output target.
errors_b4 = y_b4 - yhat_b4  # signed component errors.
print("component errors:", errors_b4)

In [ ]:
sq_b4 = errors_b4 ** 2  # squared loss per output.
print("component squared losses:", sq_b4)
assert np.allclose(sq_b4, np.array([0.25, 1.0]))
plt.figure(figsize=(4, 3))
plt.bar(["output 1", "output 2"], sq_b4, color="orange")
plt.title("Basic 4: component losses")
plt.ylabel("squared error")
plt.show()

▶ What you'll see: output 2 contributes the larger squared error.

👀 Takeaway: multi-output loss begins as one loss value per target component.

### Basic 5 — Average across q outputs

**Goal.** Turn component losses into one vector loss, because the lesson's formula divides by the number of outputs q. We build it in 2 steps.

In [ ]:
component_losses_b5 = np.array([0.25, 1.00, 0.04])  # three output losses for one example.
q_b5 = len(component_losses_b5)  # number of outputs.
print("q:", q_b5, "losses:", component_losses_b5)

In [ ]:
vector_loss_b5 = component_losses_b5.mean()  # (1/q) sum_j loss_j.
print("vector loss:", round(vector_loss_b5, 3))
assert round(vector_loss_b5, 3) == 0.430
plt.figure(figsize=(4, 3))
plt.bar(["sum", "average"], [component_losses_b5.sum(), vector_loss_b5], color=["gray", "teal"])
plt.title("Basic 5: divide by q")
plt.ylabel("loss scale")
plt.show()

▶ What you'll see: averaging keeps the loss smaller and comparable across output counts.

👀 Takeaway: dividing by q prevents a model with more outputs from looking worse merely because it has more terms.

### Basic 6 — Average losses across examples

**Goal.** Compute empirical risk from per-example losses, because ERM optimizes an average over the sample. We build it in 2 steps.

In [ ]:
losses_b6 = np.array([0.246, 0.083, 0.522])  # verified lesson losses.
print("per-example losses:", losses_b6)

In [ ]:
risk_b6 = losses_b6.mean()  # empirical risk R_S.
print("R_S:", round(risk_b6, 3))
assert round(risk_b6, 3) == 0.284
plt.figure(figsize=(4, 3))
plt.bar(["ex1", "ex2", "ex3"], losses_b6, color="steelblue")
plt.axhline(risk_b6, color="red", linestyle="--", label="mean")
plt.title("Basic 6: empirical risk")
plt.ylabel("loss")
plt.legend()
plt.show()

▶ What you'll see: the dashed mean line summarizes the three training losses.

👀 Takeaway: the training number is an average, not a hand-picked best example.

### Basic 7 — Add a complexity cost

**Goal.** Add the method cost to raw empirical risk, because the selection score must include more than fit. We build it in 2 steps.

In [ ]:
risk_b7 = 0.284  # raw empirical risk from the toy lesson.
cost_b7 = 0.080  # complexity or stability cost.
print("risk:", risk_b7, "cost:", cost_b7)

In [ ]:
score_b7 = risk_b7 + cost_b7  # full decision score.
print("score:", round(score_b7, 3))
assert round(score_b7, 3) == 0.364
plt.figure(figsize=(4, 3))
plt.bar(["risk", "cost", "score"], [risk_b7, cost_b7, score_b7], color=["teal", "orange", "gray"])
plt.title("Basic 7: risk plus cost")
plt.ylabel("value")
plt.show()

▶ What you'll see: the cost raises the decision score above the raw training risk.

👀 Takeaway: model selection should rank full scores, not raw fit alone.

### Basic 8 — Compute a validation gap

**Goal.** Compare a baseline score with a flexible alternative, because the gap is the evidence for preferring one setting. We build it in 2 steps.

In [ ]:
baseline_b8 = 0.364  # score after cost.
flexible_b8 = 0.412  # alternative decision score.
print("baseline:", baseline_b8, "flexible:", flexible_b8)

In [ ]:
gap_b8 = flexible_b8 - baseline_b8  # absolute gap.
relative_b8 = gap_b8 / flexible_b8  # scale-aware gap.
print("gap:", round(gap_b8, 3), "relative:", round(relative_b8, 3))
assert round(gap_b8, 3) == 0.048
assert round(relative_b8, 3) == 0.117
plt.figure(figsize=(4, 3))
plt.bar(["absolute", "relative"], [gap_b8, relative_b8], color="mediumpurple")
plt.title("Basic 8: model-comparison gap")
plt.ylabel("gap value")
plt.show()

▶ What you'll see: the absolute gap is 0.048 and the relative gap is about 11.7%.

👀 Takeaway: a gap only matters after you understand its scale.

### Basic 9 — Apply a stabilizing knob

**Goal.** Reduce a score by a stability factor, because constraints can improve future decisions by reducing brittle variation. We build it in 2 steps.

In [ ]:
score_b9 = 0.364  # original full score.
factor_b9 = 0.80  # a 20% reduction from stabilization.
print("score:", score_b9, "factor:", factor_b9)

In [ ]:
stable_b9 = factor_b9 * score_b9  # stabilized decision score.
print("stable score:", round(stable_b9, 3))
assert round(stable_b9, 3) == 0.291
plt.figure(figsize=(4, 3))
plt.bar(["before", "after"], [score_b9, stable_b9], color=["gray", "seagreen"])
plt.title("Basic 9: stabilizing score reduction")
plt.ylabel("decision score")
plt.show()

▶ What you'll see: the stabilized score is lower than the original score.

👀 Takeaway: regularization-like knobs trade flexibility for stability.

### Basic 10 — Choose the lowest full score

**Goal.** Select among baseline, flexible, and stabilized scores, because lower full decision score wins in this toy lesson. We build it in 2 steps.

In [ ]:
scores_b10 = np.array([0.364, 0.412, 0.291])  # baseline, flexible, stabilized.
names_b10 = np.array(["baseline", "flexible", "stabilized"])  # matching labels.
print("scores:", dict(zip(names_b10, scores_b10)))

In [ ]:
winner_b10 = names_b10[int(np.argmin(scores_b10))]  # choose the lowest full score.
print("winner:", winner_b10)
assert winner_b10 == "stabilized"
plt.figure(figsize=(4, 3))
plt.bar(names_b10, scores_b10, color=["gray", "orange", "seagreen"])
plt.title("Basic 10: final decision")
plt.ylabel("score, lower is better")
plt.xticks(rotation=15)
plt.show()

▶ What you'll see: the stabilized option has the lowest bar and wins.

👀 Takeaway: the final model choice follows the complete score implied by the method.

## 🟡 Easy

### Easy 1 — Fit a multi-output linear model

**Goal.** Solve \(Y\approx XW\) with NumPy least squares, because multi-output regression can share one design matrix across related targets. We build it in 3 steps.

In [ ]:
X_e1 = np.array([[1., 0.], [1., 1.], [1., 2.], [1., 3.]])  # intercept plus one feature.
Y_e1 = np.array([[2., 5.], [3., 4.], [4., 3.], [5., 2.]])  # two perfectly linear outputs.
print("X shape:", X_e1.shape, "Y shape:", Y_e1.shape)

In [ ]:
W_e1 = np.linalg.pinv(X_e1) @ Y_e1  # solve both output columns at once.
Yhat_e1 = X_e1 @ W_e1  # predict the full output matrix.
print("W:\n", np.round(W_e1, 3))
assert W_e1.shape == (2, 2)

In [ ]:
mse_e1 = np.mean((Y_e1 - Yhat_e1) ** 2)  # average across examples and outputs.
print("MSE:", round(float(mse_e1), 6))
assert round(float(mse_e1), 6) == 0.0
plt.figure(figsize=(5, 3))
plt.plot(X_e1[:, 1], Y_e1[:, 0], "o", label="y1")
plt.plot(X_e1[:, 1], Yhat_e1[:, 0], "-", label="fit y1")
plt.plot(X_e1[:, 1], Y_e1[:, 1], "s", label="y2")
plt.plot(X_e1[:, 1], Yhat_e1[:, 1], "--", label="fit y2")
plt.title("Easy 1: multi-output least squares")
plt.legend()
plt.show()

▶ What you'll see: two output lines are fit by one matrix equation.

👀 Takeaway: multi-output linear regression stores one coefficient column per target.

### Easy 2 — Weight outputs by importance

**Goal.** Change the average vector loss with output weights, because some target components can be more costly than others. We build it in 3 steps.

In [ ]:
sq_e2 = np.array([[0.04, 0.25, 1.00], [0.09, 0.16, 0.49]])  # squared losses for two examples and three outputs.
weights_e2 = np.array([0.2, 0.3, 0.5])  # output weights that sum to 1.
print("weights sum:", weights_e2.sum())
assert round(float(weights_e2.sum()), 3) == 1.0

In [ ]:
unweighted_e2 = sq_e2.mean(axis=1)  # ordinary 1/q average.
weighted_e2 = sq_e2 @ weights_e2  # weighted average across outputs.
print("unweighted:", np.round(unweighted_e2, 3))
print("weighted:", np.round(weighted_e2, 3))

In [ ]:
risk_e2 = weighted_e2.mean()  # average weighted vector loss over examples.
print("weighted empirical risk:", round(float(risk_e2), 3))
assert round(float(risk_e2), 3) == 0.447
plt.figure(figsize=(4, 3))
plt.bar(["ex1", "ex2"], weighted_e2, color="darkorange")
plt.title("Easy 2: weighted vector losses")
plt.ylabel("weighted loss")
plt.show()

▶ What you'll see: the third output matters most because it has the largest weight.

👀 Takeaway: output weights change the objective scale to match decision costs.

### Easy 3 — Evaluate ordinal mean absolute error

**Goal.** Compute MAE on ordered levels, because ordinal quality should reflect how far predictions move along the scale. We build it in 3 steps.

In [ ]:
truth_e3 = np.array([1, 2, 3, 4, 5, 5])  # true ordered labels.
pred_e3 = np.array([1, 3, 3, 2, 5, 4])  # predicted ordered labels.
errors_e3 = np.abs(truth_e3 - pred_e3)  # level distances.
print("errors:", errors_e3)

In [ ]:
mae_e3 = errors_e3.mean()  # mean ordinal absolute error.
within_one_e3 = np.mean(errors_e3 <= 1)  # tolerance-style ordinal accuracy.
print("MAE:", round(float(mae_e3), 3), "within one:", round(float(within_one_e3), 3))
assert round(float(mae_e3), 3) == 0.667

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(len(errors_e3)), errors_e3, color="crimson")
plt.axhline(mae_e3, color="black", linestyle="--", label="MAE")
plt.title("Easy 3: ordinal absolute errors")
plt.xlabel("example")
plt.ylabel("level distance")
plt.legend()
plt.show()

▶ What you'll see: one two-level miss dominates the average more than adjacent misses.

👀 Takeaway: ordinal MAE respects target order while staying easy to inspect.

### Easy 4 — Combine fit and cost for two models

**Goal.** Compare two candidate methods after adding costs, because the raw lower training loss may not be the selected model. We build it in 3 steps.

In [ ]:
raw_e4 = np.array([0.284, 0.260])  # raw empirical risks for stable-ish and flexible models.
cost_e4 = np.array([0.080, 0.152])  # complexity costs.
models_e4 = np.array(["structured", "flexible"])
print("raw risks:", raw_e4)

In [ ]:
scores_e4 = raw_e4 + cost_e4  # full scores.
print("scores:", dict(zip(models_e4, np.round(scores_e4, 3))))
winner_e4 = models_e4[int(np.argmin(scores_e4))]
assert winner_e4 == "structured"

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(models_e4, scores_e4, color=["seagreen", "orange"])
plt.title("Easy 4: raw fit plus cost")
plt.ylabel("full score")
plt.show()

▶ What you'll see: the flexible model has lower raw loss but higher full score after cost.

👀 Takeaway: complexity costs can reverse what raw training fit suggests.

### Easy 5 — Validate a stabilizing factor

**Goal.** Sweep a few stabilization factors, because a constraint should be chosen by validation score rather than by hope. We build it in 3 steps.

In [ ]:
base_e5 = 0.364  # original full score.
factors_e5 = np.array([1.00, 0.90, 0.80, 0.75])  # candidate stabilization strengths.
penalties_e5 = np.array([0.000, 0.010, 0.000, 0.020])  # extra costs from too little/too much constraint.
print("factors:", factors_e5)

In [ ]:
scores_e5 = base_e5 * factors_e5 + penalties_e5  # validation decision score for each factor.
best_e5 = int(np.argmin(scores_e5))
print("scores:", np.round(scores_e5, 3))
print("best factor:", factors_e5[best_e5])
assert factors_e5[best_e5] == 0.80

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(factors_e5, scores_e5, marker="o", color="navy")
plt.scatter([factors_e5[best_e5]], [scores_e5[best_e5]], color="red", zorder=3)
plt.title("Easy 5: choose stabilization by score")
plt.xlabel("stability factor")
plt.ylabel("validation score")
plt.show()

▶ What you'll see: the 0.80 factor gives the lowest validation-style score in this toy sweep.

👀 Takeaway: stabilization is a tunable model-selection knob.

## 🔴 Advanced

### Advanced 1 — Compare independent and shared multi-output fits

**Goal.** Compare separate output fits with one shared coefficient matrix, because related outputs can be trained together without losing per-output predictions. We build it in 4 steps.

In [ ]:
X_a1 = np.c_[np.ones(8), np.linspace(0, 1, 8)]  # intercept plus one feature.
Y_a1 = np.c_[2 + 3 * X_a1[:, 1], 5 - 2 * X_a1[:, 1]]  # two related linear outputs.
Y_a1 = Y_a1 + np.array([[0.00, 0.10], [0.05, -0.05], [-0.02, 0.00], [0.03, 0.04], [-0.04, -0.03], [0.02, 0.02], [0.00, -0.04], [0.01, 0.01]])
print("X shape:", X_a1.shape, "Y shape:", Y_a1.shape)

In [ ]:
W_shared_a1 = np.linalg.pinv(X_a1) @ Y_a1  # fit both outputs in one matrix operation.
Yhat_shared_a1 = X_a1 @ W_shared_a1
mse_shared_a1 = np.mean((Y_a1 - Yhat_shared_a1) ** 2)
print("shared MSE:", round(float(mse_shared_a1), 4))

In [ ]:
W_sep_a1 = []
for j_a1 in range(Y_a1.shape[1]):
    W_sep_a1.append(np.linalg.pinv(X_a1) @ Y_a1[:, j_a1])  # fit each output column separately.
W_sep_a1 = np.column_stack(W_sep_a1)
Yhat_sep_a1 = X_a1 @ W_sep_a1
mse_sep_a1 = np.mean((Y_a1 - Yhat_sep_a1) ** 2)
print("separate MSE:", round(float(mse_sep_a1), 4))
assert np.allclose(W_shared_a1, W_sep_a1)

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["shared matrix", "separate columns"], [mse_shared_a1, mse_sep_a1], color=["teal", "gray"])
plt.title("Advanced 1: same linear solution, different view")
plt.ylabel("MSE")
plt.show()

▶ What you'll see: ordinary least squares gives the same numbers, but the shared-matrix view exposes the multi-output structure.

👀 Takeaway: multi-output regression can be algebraically simple while conceptually preserving target vectors.

### Advanced 2 — Add a correlation-aware penalty

**Goal.** Penalize disagreement between related output residuals, because multi-output targets often carry joint structure beyond separate errors. We build it in 4 steps.

In [ ]:
resid_a2 = np.array([[0.2, 0.1], [-0.3, -0.2], [0.1, 0.4], [-0.1, 0.0]])  # residuals for two outputs.
base_loss_a2 = np.mean(resid_a2 ** 2)  # ordinary average squared error.
print("base loss:", round(float(base_loss_a2), 3))

In [ ]:
disagree_a2 = resid_a2[:, 0] - resid_a2[:, 1]  # difference between residual components.
penalty_a2 = 0.5 * np.mean(disagree_a2 ** 2)  # small penalty for inconsistent residual signs/magnitudes.
joint_loss_a2 = base_loss_a2 + penalty_a2
print("penalty:", round(float(penalty_a2), 3), "joint loss:", round(float(joint_loss_a2), 3))
assert round(float(joint_loss_a2), 3) == 0.060

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(resid_a2[:, 0], marker="o", label="resid output 1")
plt.plot(resid_a2[:, 1], marker="s", label="resid output 2")
plt.title("Advanced 2: residual agreement")
plt.xlabel("example")
plt.ylabel("residual")
plt.legend()
plt.show()

▶ What you'll see: examples where residual lines separate contribute to the disagreement penalty.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["base", "penalty", "joint"], [base_loss_a2, penalty_a2, joint_loss_a2], color=["gray", "orange", "teal"])
plt.title("Advanced 2: joint loss pieces")
plt.ylabel("loss")
plt.show()

▶ What you'll see: the joint loss is the ordinary fit term plus a small structure penalty.

👀 Takeaway: multi-output objectives can encode relationships among target components, not just average independent errors.

### Advanced 3 — Tune ordinal thresholds on validation data

**Goal.** Try several threshold shifts and score ordinal MAE, because cut points should match future ordered-label performance. We build it in 4 steps.

In [ ]:
score_a3 = np.array([0.8, 1.4, 1.9, 2.6, 3.2, 3.9, 4.4])  # validation latent scores.
truth_a3 = np.array([1, 1, 2, 3, 3, 4, 5])  # validation ordinal truths.
base_thresholds_a3 = np.array([1.5, 2.5, 3.5, 4.2])  # initial cut points.
shifts_a3 = np.array([-0.2, 0.0, 0.2])  # threshold shifts to validate.
print("shifts:", shifts_a3)

In [ ]:
maes_a3 = []
for shift_a3 in shifts_a3:
    th_a3 = base_thresholds_a3 + shift_a3
    pred_a3 = 1 + np.sum(score_a3[:, None] > th_a3[None, :], axis=1)
    maes_a3.append(np.mean(np.abs(truth_a3 - pred_a3)))
print("MAE by shift:", np.round(maes_a3, 3))

In [ ]:
best_idx_a3 = int(np.argmin(maes_a3))
best_shift_a3 = shifts_a3[best_idx_a3]
print("best shift:", best_shift_a3)
assert best_shift_a3 == 0.0

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(shifts_a3, maes_a3, marker="o", color="purple")
plt.axvline(best_shift_a3, color="red", linestyle="--")
plt.title("Advanced 3: threshold validation")
plt.xlabel("threshold shift")
plt.ylabel("ordinal MAE")
plt.show()

▶ What you'll see: the middle threshold setting has the lowest ordinal MAE on this validation set.

👀 Takeaway: ordinal thresholds are decision parameters and should be validated directly.

### Advanced 4 — Track train-versus-validation score

**Goal.** Show why the lesson warns against trusting training score alone, because a flexible option can win on train and lose after future-facing validation. We build it in 4 steps.

In [ ]:
models_a4 = np.array(["simple", "medium", "flexible"])
train_a4 = np.array([0.310, 0.284, 0.240])  # training risk improves with flexibility.
val_a4 = np.array([0.360, 0.364, 0.412])  # validation/full score worsens for the flexible model.
print("training risks:", dict(zip(models_a4, train_a4)))

In [ ]:
best_train_a4 = models_a4[int(np.argmin(train_a4))]
best_val_a4 = models_a4[int(np.argmin(val_a4))]
print("best by train:", best_train_a4, "best by validation/full score:", best_val_a4)
assert best_train_a4 == "flexible"
assert best_val_a4 == "simple"

In [ ]:
x_a4 = np.arange(len(models_a4))
plt.figure(figsize=(5, 3))
plt.bar(x_a4 - 0.18, train_a4, width=0.36, label="train risk", color="gray")
plt.bar(x_a4 + 0.18, val_a4, width=0.36, label="validation score", color="teal")
plt.xticks(x_a4, models_a4)
plt.title("Advanced 4: train vs future-facing score")
plt.ylabel("score")
plt.legend()
plt.show()

▶ What you'll see: the flexible model has the lowest training bar but the highest validation/full-score bar.

In [ ]:
gap_a4 = val_a4 - train_a4
print("generalization gaps:", np.round(gap_a4, 3))
assert round(float(gap_a4[-1]), 3) == 0.172

▶ What you'll see: the flexible model has the largest gap between train and validation.

👀 Takeaway: validation protects against choosing flexibility that only memorizes the sample.

### Advanced 5 — End-to-end ordinal multi-output decision

**Goal.** Combine ordinal and multi-output pieces into one final decision score, because the lesson's unit of judgment is the full method-implied score. We build it in 5 steps.

In [ ]:
Y_true_a5 = np.array([[1, 10.0], [2, 8.0], [4, 4.0], [5, 2.0]])  # first output ordinal-like, second continuous.
Y_pred_a5 = np.array([[1, 9.5], [3, 8.2], [4, 4.8], [4, 2.4]])  # candidate predictions.
print("true targets:\n", Y_true_a5)

In [ ]:
ordinal_loss_a5 = np.abs(Y_true_a5[:, 0] - Y_pred_a5[:, 0]) / 4.0  # normalized ordered error.
continuous_loss_a5 = ((Y_true_a5[:, 1] - Y_pred_a5[:, 1]) / 10.0) ** 2  # scaled squared error.
print("ordinal loss:", np.round(ordinal_loss_a5, 3))
print("continuous loss:", np.round(continuous_loss_a5, 3))

In [ ]:
vector_loss_a5 = 0.5 * ordinal_loss_a5 + 0.5 * continuous_loss_a5  # average the two output losses.
raw_a5 = float(vector_loss_a5.mean())
cost_a5 = 0.080
score_a5 = raw_a5 + cost_a5
print("raw:", round(raw_a5, 3), "score:", round(score_a5, 3))
assert round(raw_a5, 3) == 0.064

In [ ]:
stable_a5 = 0.80 * score_a5
alternative_a5 = score_a5 + 0.048
choices_a5 = np.array([score_a5, alternative_a5, stable_a5])
labels_a5 = np.array(["candidate", "flexible alt", "stabilized"])
print("choices:", dict(zip(labels_a5, np.round(choices_a5, 3))))
assert labels_a5[int(np.argmin(choices_a5))] == "stabilized"

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(labels_a5, choices_a5, color=["gray", "orange", "seagreen"])
plt.title("Advanced 5: full structured-target decision")
plt.ylabel("score, lower is better")
plt.xticks(rotation=15)
plt.show()

▶ What you'll see: the final ranking follows the complete score after vector losses, cost, and stabilization.

👀 Takeaway: structured targets change both the loss calculation and the model-selection arithmetic.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Ordinal and multi-output regression respect structure in targets instead of flattening everything into one scalar.

Ordinal and multi-output regression keeps related target coordinates together. The same average-loss move from empirical risk now averages across output dimensions as well as examples. Save a copy to Drive to edit.

In [ ]:
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_diabetes
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.datasets import make_regression
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.linear_model import Ridge
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)
np.random.seed(7)

def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...].

    All X are 2-D float feature matrices, y integer labels, so one classifier runs unchanged
    across every rung (the 'watch it scale' story). Rungs get harder: clean+separable -> real
    high-dimensional. D1 is hand-built and fully inspectable.
    """
    rungs = []

    # D1 — four hand-placed 2-D points, 2 classes, clearly separable.
    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    # D2 — clean, well-separated Gaussian blobs.
    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    # D3 — non-linear, overlapping two-moons with noise.
    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    # D4 — real: Wine, 13 features, 3 classes.
    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    # D5 — real, harder: Breast Cancer, 30 features, class imbalance.
    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs

def reg_ladder():
    """D1..D5 regression ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0], [1.0], [2.0], [3.0]])
    y1 = np.array([1.0, 3.0, 5.0, 7.0])
    rungs.append(("D1 hand line y=2x+1", x1, y1))

    rng = np.random.default_rng(1)
    x2 = np.linspace(-3, 3, 120).reshape(-1, 1)
    y2 = (2.0 * x2[:, 0] + 1.0) + rng.normal(0, 0.5, size=120)
    rungs.append(("D2 linear + noise", x2, y2))

    x3 = np.linspace(-3, 3, 160).reshape(-1, 1)
    y3 = np.sin(1.5 * x3[:, 0]) + rng.normal(0, 0.2, size=160)
    rungs.append(("D3 sine (non-linear)", x3, y3))

    dia = load_diabetes()
    rungs.append(("D4 Diabetes (real, 10-D)", dia.data, dia.target))

    x5, y5 = make_regression(n_samples=300, n_features=20, n_informative=8, noise=25.0, random_state=5)
    rungs.append(("D5 high-dim + noise (20-D)", x5, y5))

    return rungs


def lesson_score(losses, cost, alternative):
    raw = float(np.sum(losses) / len(losses))
    score = raw + cost
    gap = alternative - score
    relative_gap = gap / alternative
    return raw, score, gap, relative_gap


def preview_ladder(rungs, is_regression=False):
    rows = []
    for index, item in enumerate(rungs, start=1):
        name, X, y = item
        if is_regression:
            info = f"target range {np.min(y):.2f}..{np.max(y):.2f}"
        else:
            values, counts = np.unique(y, return_counts=True)
            pairs = [f"{int(v)}:{int(c)}" for v, c in zip(values, counts)]
            info = ", ".join(pairs)
        row = {"rung": f"D{index}", "name": name, "shape": X.shape, "info": info}
        rows.append(row)
        print(row)
    name, X, y = rungs[0]
    print("sample X:")
    print(np.round(X[:5], 3))
    print("sample y:")
    print(np.round(y[:5], 3))
    return rows


def two_dimensional_view(X):
    if X.shape[1] == 1:
        return np.c_[X[:, 0], np.zeros(X.shape[0])]
    if X.shape[1] == 2:
        return X
    view = PCA(n_components=2, random_state=0).fit_transform(StandardScaler().fit_transform(X))
    return view


def stream_batches(X, y, batch_size):
    rng = np.random.default_rng(11)
    order = rng.permutation(len(y))
    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        yield X[idx], y[idx]


def online_fit_predict(X, y, kind="sgd", epochs=8, batch_size=16):
    stratify = y if np.min(np.bincount(y.astype(int))) >= 2 else None
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.4,
        random_state=3,
        stratify=stratify,
    )
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    classes = np.unique(y)
    if kind == "pa":
        model = PassiveAggressiveClassifier(C=0.6, random_state=3, max_iter=1, tol=None)
    else:
        model = SGDClassifier(loss="log_loss", alpha=0.0005, random_state=3, learning_rate="optimal")
    first = True
    history = []
    batch_size = max(2, min(batch_size, len(y_train)))
    for epoch in range(epochs):
        for xb, yb in stream_batches(x_train, y_train, batch_size):
            if first:
                model.partial_fit(xb, yb, classes=classes)
                first = False
            else:
                model.partial_fit(xb, yb)
        preds = model.predict(x_test)
        history.append(float(accuracy_score(y_test, preds)))
    preds = model.predict(x_test)
    return model, scaler, x_train, x_test, y_train, y_test, preds, history


def logistic_accuracy(X, y, weighted=False):
    stratify = y if np.min(np.bincount(y.astype(int))) >= 2 else None
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.4,
        random_state=4,
        stratify=stratify,
    )
    class_weight = None
    if weighted:
        class_weight = "balanced"
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight=class_weight, random_state=4),
    )
    model.fit(x_train, y_train)
    preds = model.predict(x_test)
    acc = float(accuracy_score(y_test, preds))
    return model, x_train, x_test, y_train, y_test, preds, acc


def expected_binary_cost(y_true, y_pred, false_negative_cost=5.0, false_positive_cost=1.0):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return float((false_negative_cost * fn + false_positive_cost * fp) / len(y_true))


def make_multi_targets(y):
    y = np.asarray(y, dtype=float)
    scale = np.std(y)
    if scale == 0:
        scale = 1.0
    centered = (y - np.mean(y)) / scale
    cuts = np.quantile(centered, [0.33, 0.66])
    ordinal = np.digitize(centered, cuts).astype(float)
    return np.c_[centered, ordinal]


def multioutput_fit_predict(X, y, alpha=1.0):
    targets = make_multi_targets(y)
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        targets,
        test_size=0.4,
        random_state=5,
    )
    model = make_pipeline(
        StandardScaler(),
        MultiOutputRegressor(Ridge(alpha=alpha)),
    )
    model.fit(x_train, y_train)
    preds = model.predict(x_test)
    mse = float(mean_squared_error(y_test, preds))
    r2 = float(r2_score(y_test, preds, multioutput="variance_weighted"))
    return model, x_train, x_test, y_train, y_test, preds, mse, r2


def make_survival_from_classification(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    rng = np.random.default_rng(23 + X.shape[0] + X.shape[1])
    weights = np.linspace(0.4, 1.2, X.shape[1])
    linear = StandardScaler().fit_transform(X).dot(weights) / math.sqrt(X.shape[1])
    class_effect = (y == np.max(y)).astype(float) * 0.8
    risk = linear + class_effect
    event_time = np.exp(-0.45 * risk) + rng.gamma(shape=2.0, scale=0.25, size=len(y))
    censor_time = rng.gamma(shape=2.3, scale=0.5, size=len(y)) + 0.35
    observed_time = np.minimum(event_time, censor_time)
    event = (event_time <= censor_time).astype(int)
    if np.sum(event) < 3:
        event[:3] = 1
    return observed_time, event


def cox_fit(X, time, event, lr=0.03, steps=220, l2=0.02):
    X = np.asarray(X, dtype=float)
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    beta = np.zeros(X.shape[1])
    order = np.argsort(-time)
    X_desc = X[order]
    event_desc = event[order]
    for step in range(steps):
        scores = np.clip(X_desc.dot(beta), -30, 30)
        exp_scores = np.exp(scores)
        risk_sum = np.cumsum(exp_scores)
        weighted_sum = np.cumsum(exp_scores[:, None] * X_desc, axis=0)
        grad = np.zeros_like(beta)
        event_positions = np.where(event_desc == 1)[0]
        for pos in event_positions:
            grad += X_desc[pos] - weighted_sum[pos] / risk_sum[pos]
        grad = grad / max(1, len(event_positions))
        grad = grad - l2 * beta
        beta = beta + lr * grad
    return beta


def concordance_index(time, event, risk):
    total = 0
    good = 0.0
    for i in range(len(time)):
        for j in range(len(time)):
            if time[i] < time[j] and event[i] == 1:
                total += 1
                if risk[i] > risk[j]:
                    good += 1.0
                elif risk[i] == risk[j]:
                    good += 0.5
    if total == 0:
        return 0.5
    return float(good / total)


def survival_fit_score(X, y):
    time, event = make_survival_from_classification(X, y)
    stratify = y if np.min(np.bincount(y.astype(int))) >= 2 else None
    x_train, x_test, t_train, t_test, e_train, e_test = train_test_split(
        X,
        time,
        event,
        test_size=0.4,
        random_state=6,
        stratify=stratify,
    )
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    beta = cox_fit(x_train, t_train, e_train)
    train_risk = x_train.dot(beta)
    test_risk = x_test.dot(beta)
    cindex = concordance_index(t_test, e_test, test_risk)
    return beta, scaler, x_train, x_test, t_train, t_test, e_train, e_test, train_risk, test_risk, cindex


def cross_validation_gap(X, y, k=5):
    counts = np.bincount(y.astype(int))
    min_count = int(np.min(counts))
    k = max(2, min(k, min_count))
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=8),
    )
    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=8)
    result = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
    )
    train_loss = 1.0 - result["train_score"]
    val_loss = 1.0 - result["test_score"]
    gap = float(np.mean(val_loss - train_loss))
    return train_loss, val_loss, gap, cv

## The concept, built once (D1)

The lesson formula is

$$\hat y=(\hat y_1,\ldots,\hat y_q),\qquad L=\frac1q\sum_{j=1}^q \ell(y_j,\hat y_j)$$

Plug in the lesson losses 0.246, 0.083, and 0.522. The average is $R_S=0.851/3=0.284$, the cost is $0.080$, the score is $0.364$, and the alternative gap is $0.412-0.364=0.048$.

In [ ]:
def ordinal_multi_output_regression_method():
    losses = np.array([0.246, 0.083, 0.522], dtype=float)
    cost = 0.080
    alternative = 0.412
    true_outputs = np.array([2.0, 1.0, 0.0])
    predicted_outputs = np.array([1.5, 1.2, 0.7])
    per_output_squared_loss = (true_outputs - predicted_outputs) ** 2
    multi_output_loss = float(np.mean(per_output_squared_loss))
    raw, score, gap, relative_gap = lesson_score(losses, cost, alternative)
    assert np.isclose(multi_output_loss, (0.25 + 0.04 + 0.49) / 3.0)
    assert np.isclose(raw, 0.283666666667)
    assert np.isclose(score, 0.363666666667)
    assert np.isclose(gap, 0.048333333333)
    return {"per_output_squared_loss": per_output_squared_loss, "multi_output_loss": multi_output_loss, "raw": raw, "score": score, "gap": gap}

lesson_check = ordinal_multi_output_regression_method()
print(lesson_check)

The method returns the arithmetic pieces and asserts the exact lesson numbers before any larger data appears.

In [ ]:
assert lesson_check['score'] > lesson_check['raw']
assert lesson_check['gap'] > 0
print('lesson arithmetic locked')

## The dataset ladder

Use the shared regression ladder so the same multi-output regressor runs from a hand line to a high-dimensional noisy regression problem.

In [ ]:
rungs = reg_ladder()
ladder_preview = preview_ladder(rungs, is_regression=True)

## Run the same method across D1–D5

Only the data rung changes. The metric is the plan metric for this topic.

In [ ]:
results = []
artifacts = []
for rung_index, (name, X, y) in enumerate(rungs, start=1):
    model, x_train, x_test, y_train, y_test, preds, mse, r2 = multioutput_fit_predict(X, y)
    results.append({"rung": rung_index, "name": name, "mse": mse, "r2": r2})
    artifacts.append((name, X, y, y_test, preds, model))
for row in results:
    print(f"D{row['rung']} {row['mse']:.3f} MSE, R2={row['r2']:.3f} — {row['name']}")

## Results visualization

The first figure shows the model artifact on each rung. The second summarizes `mse` from D1 through D5.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))
for index, artifact in enumerate(artifacts):
    name, X, y, y_test, preds, model = artifact
    axes[index].scatter(y_test[:, 0], preds[:, 0], s=18, alpha=0.75)
    low = min(float(np.min(y_test[:, 0])), float(np.min(preds[:, 0])))
    high = max(float(np.max(y_test[:, 0])), float(np.max(preds[:, 0])))
    axes[index].plot([low, high], [low, high], color="black", linewidth=1)
    axes[index].set_title(f"D{index + 1}: predicted output 1")
    axes[index].set_xlabel("true")
    axes[index].set_ylabel("predicted")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3.5))
plt.plot([row["rung"] for row in results], [row["mse"] for row in results], marker="o")
plt.xticks([1, 2, 3, 4, 5], ["D1", "D2", "D3", "D4", "D5"])
plt.ylabel("MSE")
plt.title("multi-output MSE vs. ladder complexity")
plt.grid(True, alpha=0.3)
plt.show()

## Pitfall on the hardest rung

The lesson warning is to optimize the raw term and forget the cost. On D5, the raw metric alone can pick a different setting than the cost-aware score.

In [ ]:
name, X, y = rungs[-1]
complex_fit = multioutput_fit_predict(X, y, alpha=0.001)
stable_fit = multioutput_fit_predict(X, y, alpha=100.0)
complex_mse = complex_fit[6]
stable_mse = stable_fit[6]
raw_only_winner = "complex" if complex_mse <= stable_mse else "stable"
complex_score = complex_mse + 0.080 * 2.0
stable_score = stable_mse + 0.080 * 0.2
cost_aware_winner = "complex" if complex_score <= stable_score else "stable"
print("D5 raw MSE", complex_mse, stable_mse, "raw winner", raw_only_winner)
print("D5 cost-aware scores", complex_score, stable_score, "cost-aware winner", cost_aware_winner)
print("lesson raw", 0.284, "cost", 0.080, "score", 0.364, "gap", 0.412 - 0.364)

## Evaluate it + Practice

- Compare the displayed metric with a no-skill baseline such as majority class, mean target, or random fold assignment.
- Sanity-check D1 by hand before trusting the D5 curve.
- Ablate the key idea: remove partial updates, remove PA margins, collapse outputs, ignore censoring, remove costs, or reuse the test set.
- Failure signals include unstable D5 metrics, a widening validation gap, or a cost-aware score that disagrees with the raw metric.

Practice 1: change the seed or batch/fold size and rerun the D1-to-D5 table.

Practice 2: turn off the topic-specific idea and measure the metric drop on D5.

Practice 3: add one extra diagnostic plot for the hardest rung.